### Q1. You are working on a machine learning project where you have a dataset containing numerical and categorical features. You have identified that some of the features are highly correlated and there are missing values in some of the columns. You want to build a pipeline that automates the feature engineering process and handles the missing values.

Design a pipeline that includes the following steps:

* Use an automated feature selection method to identify the important features in the dataset.
* Create a numerical pipeline that includes the following steps:
  * Impute the missing values in the numerical columns using the mean of the column values.
  * Scale the numerical columns using standardisation.
* Create a categorical pipeline that includes the following steps:
  * Impute the missing values in the categorical columns using the most frequent value of the column
  * One-hot encode the categorical columns
  * Combine the numerical and categorical
  pipelines using a ColumnTransformer.
  * Use a Random Forest Classifier to build the final model
  * Evaluate the accuracy of the model on the test dataset.

Note: Your solution should include code snippets for each step of the pipeline, and a brief explanation of
each step. You should also provide an interpretation of the results and suggest possible improvements for
the pipeline.

Ans:







In [1]:
# Step 1: Import Libraries
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

from sklearn.metrics import accuracy_score

These libraries help build preprocessing pipelines, feature selection, and modeling.

In [3]:
# Step 2: Load Data & Split

# Example
df = pd.read_csv("data.csv")

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

We split data into training (80%) and testing (20%).

In [4]:
# Step 3: Identify Column Types
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

Separating numerical and categorical columns is essential for different preprocessing.

In [5]:
# Step 4: Numerical Pipeline
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),   # handle missing values
    ("scaler", StandardScaler())                  # standardization
])

✔ Mean imputation replaces missing values

✔ StandardScaler ensures features have mean=0, std=1

In [6]:
# Step 5: Categorical Pipeline
cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),  # fill missing
    ("encoder", OneHotEncoder(handle_unknown="ignore"))    # convert to numeric
])

✔ Missing values replaced with most common category

✔ One-hot encoding converts categories → numeric vectors

In [7]:
# Step 6: Combine Pipelines
preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

This applies:

* numerical pipeline → numeric columns

* categorical pipeline → categorical columns

In [8]:
# Step 7: Feature Selection (Automated)
feature_selector = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=42)
)

Uses feature importance from Random Forest to:

* remove less important features

* reduce multicollinearity

In [9]:
# Step 8: Final Pipeline with Model
model_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("feature_selection", feature_selector),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

Full pipeline:

* Preprocessing
* Feature selection
* Model training

In [10]:
# Step 9: Train Model
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['age', 'salary', 'experience'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['department', 'education'], dtype='object'))])),
                ('feature_selection',
                 SelectFromModel(estimator=RandomForestClassifier(random_state=42))),
                ('classifier', RandomForestClassifier(random_state=42))])

In [11]:
# Step 10: Evaluate Model
y_pred = model_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 1.0


Accuracy measures:

Accuracy=Correct Predictions/
Total Predictions


📊 Interpretation of Results:

* If accuracy is high (e.g., >85%) → model is performing well
* If accuracy is low:
 * data may be noisy
 * features may not be informative
 * model may be underfitting

Also:

* Random Forest handles correlated features better than linear models
* Feature selection helps remove redundant features


# 🚀 Possible Improvements

### 1. Hyperparameter Tuning

```python
from sklearn.model_selection import GridSearchCV
```

Tune:

* `n_estimators`
* `max_depth`
* `min_samples_split`

---

### 2. Use Cross Validation

```python
from sklearn.model_selection import cross_val_score
```

👉 Gives more reliable performance than a single split

---

### 3. Handle Imbalanced Data

* Use:

  * `class_weight="balanced"`
  * SMOTE (oversampling)

---

### 4. Advanced Feature Selection

* Try:

  * Recursive Feature Elimination (RFE)
  * Mutual Information

---

### 5. Try Other Models

* Gradient Boosting
* XGBoost
* Logistic Regression (for interpretability)

---

# 🧠 Final Summary

This pipeline:

✔ Handles missing values

✔ Encodes categorical data

✔ Scales numerical data

✔ Removes irrelevant features

✔ Trains a robust model

👉 Most important advantage:
**Fully automated and reusable pipeline** (no manual preprocessing needed)




### Q2. Build a pipeline that includes a random forest classifier and a logistic regression classifier, and then use a voting classifier to combine their predictions. Train the pipeline on the iris dataset and evaluate its accuracy.

Ans:



In [12]:
# Step 1: Import Libraries
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

In [13]:
# Step 2: Load Dataset
iris = load_iris()

X = iris.data
y = iris.target

The Iris dataset contains:

* 150 samples
* 4 numerical features (sepal & petal measurements)
* 3 classes (species)

In [14]:
# Step 3: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
# Step 4: Define Base Models
rf = RandomForestClassifier(n_estimators=100, random_state=42)

lr = LogisticRegression(max_iter=200)

* Random Forest → handles non-linearity
* Logistic Regression → simple & interpretable

In [16]:
# Step 5: Create Voting Classifier
voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('lr', lr)
    ],
    voting='hard'   # majority voting
)

* Hard voting → majority class wins
* You can also use voting='soft' (probability-based)

In [17]:
# Step 6: Build Pipeline
pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),   # important for Logistic Regression
    ("voting", voting_clf)
])

* Scaling improves Logistic Regression performance.

In [18]:
# Step 7: Train Model
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('voting',
                 VotingClassifier(estimators=[('rf',
                                               RandomForestClassifier(random_state=42)),
                                              ('lr',
                                               LogisticRegression(max_iter=200))]))])

In [19]:
# Step 8: Evaluate Accuracy
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 1.0



# 🧠 Interpretation

* The Voting Classifier combines strengths of:

  * Random Forest → captures complex patterns
  * Logistic Regression → provides stable linear boundaries

👉 Result:

* Better generalization
* Reduced overfitting compared to a single model

---

# 🚀 Improvements

### 1. Use Soft Voting (Better Performance)

```python
voting='soft'
```

---

### 2. Add More Models

* SVM
* KNN
* Gradient Boosting

---

### 3. Hyperparameter Tuning

Use:

```python
GridSearchCV
```

---

### 4. Cross Validation

```python
from sklearn.model_selection import cross_val_score
```

---

# ✅ Final Summary

✔ Built ensemble pipeline

✔ Combined two models using voting

✔ Achieved high accuracy

✔ Easy to extend and improve

